# 03 - Sistema Especialista baseado em Regras SE-ENTÃO

**Objetivo:** Criar base de conhecimento (regras SE-ENTÃO) separada do motor de inferência, medir **esforço humano** com cronômetro e **eficiência computacional apenas na inferência**.

**Regra:** elaborar usando **apenas o conjunto de treino**.


In [ ]:
# Configuração
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import time, tracemalloc
from datetime import datetime
from src.config import SEED, CLASSIFICATION_TARGET, REGRESSION_TARGET
from src.data import split_train_test
np.random.seed(SEED)


## 1) Carregamento e divisão

In [ ]:
df_reg = pd.read_csv('../data/processed/dataset_regression.csv')
df_clf = pd.read_csv('../data/processed/dataset_classification.csv')
Xr_train, Xr_test, yr_train, yr_test = split_train_test(df_reg, REGRESSION_TARGET, stratify=False)
Xc_train, Xc_test, yc_train, yc_test = split_train_test(df_clf, CLASSIFICATION_TARGET, stratify=True)
print(Xr_train.shape, Xr_test.shape, Xc_train.shape, Xc_test.shape)


## 2) Base de Conhecimento

**Tipos:** relação, recomendação, diretriz, estratégia, heurística. **Não misturar E e OU na mesma regra.**

In [ ]:
# Preencha suas regras aqui
KB_CLF = []  # [{'id':1,'tipo':'heurística','se':'...', 'entao':'Burnout_Risk=High','justificativa':'...'}]
KB_REG = []
print('Total regras clf:', len(KB_CLF), '| reg:', len(KB_REG))


## 3) Cronometragem (mínimo 3 sessões)

In [ ]:
class Cronometro:
    def __init__(self): self.i=None; self.f=None
    def iniciar(self): self.i=time.perf_counter(); print('INICIO', datetime.now().isoformat())
    def parar(self): self.f=time.perf_counter(); print('FIM', datetime.now().isoformat()); print(f'{(self.f-self.i)/60:.2f} min')

cr = Cronometro()
# cr.iniciar() / cr.parar()


## 4) Motor de Inferência + Medição

In [ ]:
def medir_inferencia(func, X, n_rep=30):
    tempos=[]; tracemalloc.start()
    for _ in range(n_rep):
        t0=time.perf_counter(); func(X); tempos.append(time.perf_counter()-t0)
    _,pico=tracemalloc.get_traced_memory(); tracemalloc.stop()
    m=np.mean(tempos); s=np.std(tempos)
    return {'tempo_medio_s':m,'tempo_desvio_s':s,'tempo_por_amostra_ms':1000*m/len(X),'vazao_amostras_s':len(X)/m,'pico_memoria_kb':pico/1024}
